In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
model_idx = 7

In [3]:
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
# from train_model import HiCDataset
# from model import SeqNN
from model_v2_compatible import SeqNN
# from model_v2_horiz_checkpoint import SeqNN

In [4]:
from torch.utils.data import DataLoader, Dataset

In [ ]:
from data_processing.dataset import HiCDataset

In [5]:
class HiCDataset(Dataset):
    def __init__(self, data_files):
        self.data = []  # To store all sequences and HiC vectors
        
        # Load and process the data files
        for file in data_files:
            print("Loading file:", file)
            file_data = torch.load(file, weights_only=True)
            
            for data in file_data:
                ohe_sequence, hic_vector = data

                # Process the OHE sequence
                ohe_sequence = ohe_sequence.squeeze(0)  # Remove singleton dimension
                
                # Ensure the sequence has the correct shape
                assert ohe_sequence.shape[0] == 4, f"Expected 4 channels, but got {ohe_sequence.shape[0]}"
                assert len(ohe_sequence.shape) == 2, f"Expected 2D shape for sequence, but got {ohe_sequence.shape}"
                
                # Add processed pair to the data list
                self.data.append((ohe_sequence, hic_vector))
        
        print(f"Total sequences loaded: {len(self.data)}")
        
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Fetch the preprocessed (ohe_sequence, hic_vector) pair from memory
        ohe_sequence, hic_vector = self.data[idx]
        return ohe_sequence, hic_vector

In [6]:
torch.serialization.add_safe_globals([SeqNN])

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
model_path = f"/scratch1/smaruj/Akita_pytorch_models/finetuned/mouse_models/" \
                 f"Vian2018_Bcells/models/Akita_v2_mouse_Vian2018_Bcells_model{model_idx}_R_from_scratch.pth"

In [9]:
model = SeqNN()
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 128, kernel_size=(15,), stride=(1,), padding=(7,), bias=False)
    (batch_norm): BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size=2, stride=2, paddi

In [ ]:
# model = SeqNN()
# model = torch.load(model_path)
# model.eval()

In [ ]:
# model = SeqNN()
# # the entire model
# model.load_state_dict(torch.load(model_path, map_location=device))
# # model.load_state_dict(torch.load(f"/scratch1/smaruj/Akita_pytorch_models/finetuned/mouse_models/Hsieh2019_mESC/models/Akita_v2_mouse_Hsieh2019_mESC_model{model_idx}_finetuned.pth", map_location=device))
# # saved weights only
# # model = torch.load(f"/scratch1/smaruj/Akita_pytorch_models/tf_transferred/human_models/Krietenstein2019_HFF/Akita_v2_human_Krietenstein2019_HFF_model{model_idx}.pth")
# # model = torch.load(f"/scratch1/smaruj/Akita_pytorch_models/finetuned/mouse_models/Hsieh2019_mESC/models/Akita_v2_mouse_Hsieh2019_mESC_model{model_idx}_finetuned.pth")
# # model.to(device)
# model.eval()

In [ ]:
# may be needed
# state_dict = torch.load("/scratch1/smaruj/train_pytorch_akita/mouse_models/a100_2gpus_memory_opt_finetuning.pt")
# from collections import OrderedDict
# new_state_dict = OrderedDict((k.replace("module.", ""), v) for k, v in state_dict.items())
# model.load_state_dict(new_state_dict)

In [ ]:
# model.to(device)
# model.eval()

In [10]:
import os

In [11]:
data_dir = "/scratch1/smaruj/Akita_pytorch_training_data/mouse_data/Vian2018_Bcells"
test_fold = f"fold{model_idx}"

all_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".pt")]
test_files = [f for f in all_files if test_fold in f]

In [12]:
test_dataset = HiCDataset(test_files)

Loading file: /scratch1/smaruj/Akita_pytorch_training_data/mouse_data/Vian2018_Bcells/fold7_5.pt
Loading file: /scratch1/smaruj/Akita_pytorch_training_data/mouse_data/Vian2018_Bcells/fold7_4.pt
Loading file: /scratch1/smaruj/Akita_pytorch_training_data/mouse_data/Vian2018_Bcells/fold7_6.pt
Loading file: /scratch1/smaruj/Akita_pytorch_training_data/mouse_data/Vian2018_Bcells/fold7_7.pt
Loading file: /scratch1/smaruj/Akita_pytorch_training_data/mouse_data/Vian2018_Bcells/fold7_3.pt
Loading file: /scratch1/smaruj/Akita_pytorch_training_data/mouse_data/Vian2018_Bcells/fold7_2.pt
Loading file: /scratch1/smaruj/Akita_pytorch_training_data/mouse_data/Vian2018_Bcells/fold7_0.pt
Loading file: /scratch1/smaruj/Akita_pytorch_training_data/mouse_data/Vian2018_Bcells/fold7_1.pt
Total sequences loaded: 714


In [13]:
# Load test dataset
# test_dataset = HiCDataset(test_files)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
# Run model on test set
# scd_values = []

# all_preds = []
# all_targets = []

# with torch.no_grad():
#     for ohe_sequence, hic_vector in test_loader:
#         ohe_sequence = ohe_sequence.to(device)
#         hic_vector = hic_vector.to(device)
        
#         outputs = model(ohe_sequence)
#         all_preds.append(outputs.cpu())
#         all_targets.append(hic_vector.cpu())
        
#         # Compute SCD
#         scd_batch = torch.sqrt((outputs ** 2).sum(dim=(1, 2)))  # [B]
#         scd_values.extend(scd_batch.cpu().numpy())

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def plot_map(matrix, vmin=-0.6, vmax=0.6, palette="RdBu_r", width=5, height=5):
    fig, axes = plt.subplots(1, 1, figsize=(width, height))

    sns.heatmap(
        matrix,
        vmin=vmin,
        vmax=vmax,
        cbar=False,
        cmap=palette,
        square=True,
        xticklabels=False,
        yticklabels=False,
        ax=axes
    )

    plt.tight_layout()
    plt.show()
    

# Helper function to set diagonal elements to a specific value
def set_diag(matrix, value, k):
    # Explicitly set the diagonal to 'value' (in this case, np.nan) for each k
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value


def from_upper_triu(vector_repr, matrix_len, num_diags):
    # Ensure vector_repr is a NumPy array (if it's a PyTorch tensor, convert it)
    if isinstance(vector_repr, torch.Tensor):
        vector_repr = vector_repr.detach().flatten().cpu().numpy()  # Flatten and convert to NumPy array

    # Initialize a zero matrix of shape (matrix_len, matrix_len)
    z = np.zeros((matrix_len, matrix_len))

    # Get the indices for the upper triangular matrix
    triu_tup = np.triu_indices(matrix_len, num_diags)

    # Assign the values from the vector_repr to the upper triangular part of the matrix
    z[triu_tup] = vector_repr

    # Set the diagonals specified by num_diags to np.nan
    for i in range(-num_diags + 1, num_diags):
        set_diag(z, np.nan, i)

    # Ensure the matrix is symmetric
    return z + z.T

In [ ]:
# from scipy.stats import pearsonr, spearmanr

In [ ]:
# # --- Stack arrays ---
# all_preds = np.concatenate(all_preds, axis=0)
# all_targets = np.concatenate(all_targets, axis=0)

In [ ]:
# Convert lists to numpy arrays for convenience
# pearson_arr = np.array(pearson_list)
# spearman_arr = np.array(spearman_list)
# mse_arr = np.array(mse_list)

In [ ]:
# # Set up figure with 3 subplots
# fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# # Histogram for PearsonR
# axes[0].hist(pearson_arr[~np.isnan(pearson_arr)], bins=30, color='skyblue', edgecolor='black')
# axes[0].set_title('PearsonR Distribution')
# axes[0].set_xlabel('PearsonR')
# axes[0].set_ylabel('Count')

# # Histogram for SpearmanR
# axes[1].hist(spearman_arr[~np.isnan(spearman_arr)], bins=30, color='lightgreen', edgecolor='black')
# axes[1].set_title('SpearmanR Distribution')
# axes[1].set_xlabel('SpearmanR')
# axes[1].set_ylabel('Count')

# # Histogram for MSE
# axes[2].hist(mse_arr, bins=30, color='salmon', edgecolor='black')
# axes[2].set_title('MSE Distribution')
# axes[2].set_xlabel('MSE')
# axes[2].set_ylabel('Count')

# plt.tight_layout()
# plt.show()

In [ ]:
# import random

In [ ]:
# # --- Show 1/10 of examples as maps ---
# num_examples = len(all_preds)
# subset_indices = random.sample(range(num_examples), k=max(1, num_examples // 10))

In [ ]:
# for idx in subset_indices:
#     # Get the correct single vector from all_preds / all_targets
#     true_vec = all_targets[idx].reshape(-1)
#     pred_vec = all_preds[idx].reshape(-1)
    
#     true_map = from_upper_triu(true_vec, matrix_len=512, num_diags=2)
#     pred_map = from_upper_triu(pred_vec, matrix_len=512, num_diags=2)

#     print(f"Example {idx}")
#     print(f"PearsonR: {pearson_list[idx]:.3f}, "
#           f"SpearmanR: {spearman_list[idx]:.3f}, "
#           f"MSE: {mse_list[idx]:.4f}")
    
#     fig, axs = plt.subplots(1, 2, figsize=(12, 5))
#     axs[0].imshow(true_map, cmap='RdBu_r', vmin=-0.6, vmax=0.6)
#     axs[0].set_title("Target")
    
#     axs[1].imshow(pred_map, cmap='RdBu_r', vmin=-0.6, vmax=0.6)
#     axs[1].set_title(f"Prediction\nPearsonR: {pearson_list[idx]:.3f}\nSpearmanR: {spearman_list[idx]:.3f}\nMSE: {mse_list[idx]:.4f}")
    
#     for ax in axs:
#         ax.axis('off')  # Hide axes for cleaner visualization
    
#     plt.tight_layout()
#     plt.show()

In [14]:
# Run model on test set
scd_values = []

all_preds = []
all_targets = []

with torch.no_grad():
    for ohe_sequence, hic_vector in test_loader:
        ohe_sequence = ohe_sequence.to(device)
        hic_vector = hic_vector.to(device)
        
        outputs = model(ohe_sequence)
        all_preds.append(outputs.cpu())
        all_targets.append(hic_vector.cpu())
        
        # Compute SCD
        scd_batch = torch.sqrt((outputs ** 2).sum(dim=(1, 2)))  # [B]
        scd_values.extend(scd_batch.cpu().numpy())

In [15]:
# Convert lists to tensors
all_preds = torch.cat(all_preds, dim=0)
all_targets = torch.cat(all_targets, dim=0)

In [ ]:
num_examples = 15
for i in range(num_examples):
    print("Example", i)
    print("Target")
    plot_map(from_upper_triu(all_targets[i, :, :], matrix_len=512, num_diags=2))
    print("Prediction")
    plot_map(from_upper_triu(all_preds[i, :], matrix_len=512, num_diags=2))

In [16]:
from scipy.stats import pearsonr, spearmanr

In [17]:
pearson_corr, _ = pearsonr(all_targets.flatten(), all_preds.flatten())

In [18]:
print(f"Pearson correlation: {pearson_corr:.4f}")

Pearson correlation: 0.6192


In [19]:
spearman_corr, _ = spearmanr(all_targets.flatten(), all_preds.flatten())

In [20]:
print(f"Spearman's R (flattened across all sequences): {spearman_corr:.4f}")

Spearman's R (flattened across all sequences): 0.5256


In [21]:
from sklearn.metrics import mean_squared_error

In [22]:
mse = mean_squared_error(all_targets.flatten(), all_preds.flatten())

In [23]:
print(f"MSE: {mse:.5f}")

MSE: 0.11122
